# M6: Descomposición híbrida LNS para N=10 (y N>=14)

Según el plan: búsqueda de vecindarios grandes (LNS) con subsolver QAOA.

**Idea:**
1. Partir de una solución inicial (heurística clásica o aleatoria, con semilla)
2. Repetir T veces (~30): elegir un vecindario S con k vuelos (k=3 → 9 qubits, k=4 → 16 qubits). Elegir aleatoriamente o vuelo + sus vecinos con mayor flujo.
3. Fijar las puertas de los vuelos fuera de S, optimizar solo S (subproblema QUBO de tamaño kxk con términos de interacción con el resto).
4. Resolver el subproblema con QAOA (Camino B) y aceptar si el costo global mejora (evaluado siempre con `cost()`).
5. **Ablación:** resolver los mismos subproblemas por enumeración exacta (k! ≤ 24). Se espera que QAOA iguale ese resultado.

**Métricas:** costo final, brecha vs 340, número de vecindarios mejorados, fracción factible por subproblema, tiempo total.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
from qap.quantum import solve_hybrid_lns
from qap.data import load_instance
from qap.types import cost

## Estado actual

`solve_hybrid_lns` está implementado como `NotImplementedError`. Este notebook documenta el diseño según el plan M6, listo para cuando se implemente.

In [ ]:
# try:
#     inst = load_instance(10)
#     res = solve_hybrid_lns(inst, seed=0, k=4, T=30, reps=1, maxiter=80, shots=2048)
#     print(res)
# except Exception as e:
#     print(e)

## Diseño del algoritmo (pseudo-código)

```python
# start: mejor heurística o permutación aleatoria
p = best_of_heuristic_or_random(seed)
for t in range(T):
    # elegir vecindario S
    S = sample_neighborhood(p, k, strategy='random_or_high_flow')
    G = {p[i] for i in S}  # puertas usadas
    # construir subproblema kxk
    # cost(y) = interacciones dentro S + interacciones con vuelos fuera (fijos)
    # resolver con QAOA (Camino B)
    p' = solve_subqaoa(S, G, p, inst, seed=...)
    if cost(p', inst) < cost(p, inst):
        p = p'
```

**Ablación:** para cada subproblema, enumerar todas las asignaciones k! y quedarse con la que minimice el costo del subproblema/evaluación global correspondiente. Comparar con QAOA.